<a href="https://colab.research.google.com/github/Brigada-Inginerilor-Amarati/Prelucrarea-Imaginilor/blob/main/Laborator/Teorie/Notebooks/PI_Laborator_11_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Video classification

*Video classification* represents the task of identifying actions in videos, as opposed to image classification, where only the most important object in the image has to be identified. In video classification, the models have access to not only the appearance information present in single, static images, but also their complex temporal evolution.

Unlike images, which can be cropped and rescaled to a fixed size, videos vary widely in temporal extent and cannot be easily processed with a fixed-sized architecture. Thus, every video is treated as a bag of short, fixed-sized clips.
Since each clip contains several contiguous frames in time, we can extend the connectivity of the network in the time dimension to learn spatio-temporal features. One way of achieving this is through 3D convolutions and 3D pooling.

We start by installing PyAV, a library needed in order for the video module in PyTorch to work, and then adding the necessary imports.

In [ ]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.0/33.0 MB 16.9 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import random_split, DataLoader
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision.models.video import r3d_18
from torchvision import transforms
import os
import warnings

torch.manual_seed(42);

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

The dataset that we will be using is [HMDB51](https://serre-lab.clps.brown.edu/resource/hmdb-a-large-human-motion-database/) (Human Motion Database 51). HMDB51 is an action recognition dataset, collected from various sources, mostly from movies, and a small proportion from public databases such as the Prelinger archive, YouTube, and Google videos. The dataset contains over $7000$ clips divided into $51$ action categories, each containing a minimum of $100$ clips. Let's download the HMDB51 dataset and the train/test splits.

Go to https://serre.lab.brown.edu/hmdb51.html, and download the HMDB51 (official release) as well as the Train/Test splits (3 lists). Locate the HMDB51 archive files that you downloaded on your drive.

Now, we extract and organize the video data into folders and subfolders, as is specified by the HMDB51 dataset.

In [ ]:
!mkdir -p video_data test_train_splits
!unrar e 'drive/MyDrive/.../test_train_splits.rar' test_train_splits # Provide the proper path location
!unrar e 'drive/MyDrive/.../hmdb51_org.rar' video_data # Provide the proper path location

for files in os.listdir('video_data'):
    foldername = files.split('.')[0]
    os.system("mkdir -p video_data/" + foldername)
    os.system("unrar e video_data/"+ files + " video_data/" + foldername)
!rm video_data/*.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from test_train_splits.rar

Extracting  test_train_splits/brush_hair_test_split1.txt                   0%  OK 
Extracting  test_train_splits/brush_hair_test_split2.txt                   1%  OK 
Extracting  test_train_splits/brush_hair_test_split3.txt                   1%  OK 
Extracting  test_train_splits/cartwheel_test_split1.txt                    2%  OK 
Extracting  test_train_splits/cartwheel_test_split2.txt                    2%  OK 
Extracting  test_train_splits/cartwheel_test_split3.txt                    3%  OK 
Extracting  test_train_splits/catch_test_split1.txt                        4%  OK 
Extracting  test_train_splits/catch_test_split2.txt                        4%  OK 
Extracting  test_train_splits/catch_test_split3.txt                        5%  OK 
Extracting  test_train_splits/chew_test_split1.txt       

Next, we build the training, validation, and testing datasets and dataloders, which will be used for model training. The input resolution will be $112\times 112$.

In [ ]:
val_split = 0.05 #3 min
num_frames = 16
clip_steps = 50
num_workers = 8
bs = 4

train_tfms = torchvision.transforms.Compose([
                                 transforms.Lambda(lambda x: x.permute(3, 0, 1, 2).to(torch.float32) / 255.),
                                 transforms.Resize((128, 171)),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.RandomCrop((112, 112))
                               ])
test_tfms = torchvision.transforms.Compose([
                                             transforms.Lambda(lambda x: x.permute(3, 0, 1, 2).to(torch.float32) / 255.),
                                             transforms.Resize((128, 171)),
                                             transforms.CenterCrop((112, 112))
                                             ])

hmdb51_train = torchvision.datasets.HMDB51('video_data/', 'test_train_splits/', num_frames,
                                                step_between_clips=clip_steps, fold=1, train=True,
                                                transform=train_tfms, num_workers=num_workers)

hmdb51_test = torchvision.datasets.HMDB51('video_data/', 'test_train_splits/', num_frames,
                                                step_between_clips=clip_steps, fold=1, train=False,
                                                transform=test_tfms, num_workers=num_workers)


total_train_samples = len(hmdb51_train)
total_val_samples = round(val_split * total_train_samples)

hmdb51_train_v1, hmdb51_val_v1 = random_split(hmdb51_train, [total_train_samples - total_val_samples, total_val_samples])

train_loader = DataLoader(hmdb51_train_v1, batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(hmdb51_val_v1, batch_size=bs, shuffle=True, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(hmdb51_test, batch_size=bs, shuffle=False, num_workers=num_workers, pin_memory=True)

print(f"number of train samples {total_train_samples - total_val_samples}")
print(f"number of validation samples {total_val_samples}")
print(f"number of test samples {len(hmdb51_test)}")

100%|██████████| 423/423 [01:48<00:00,  3.89it/s]


number of train samples 7366
number of validation samples 388
number of test samples 3234


We can now define the model that will be used for video classification. In this case, it will be the `r3d_18` network which is already defined in PyTorch. It is the 3D extension of the ResNet-18 network, which is called ResNet3D-18. We exclude the last linear layer of the network, since it does not match the number of classes in the HMDB51 dataset.

In [ ]:
class VideoRecognitionModel(nn.Module):
  def __init__(self):
      super(VideoRecognitionModel, self).__init__()
      self.base_model = nn.Sequential(*list(r3d_18().children())[:-1])
      self.fc = nn.Linear(512, 51)

  def forward(self, x):
      out = self.base_model(x).flatten(1)
      out = F.relu(self.fc(out))
      return out

The `train` function is similar to any such function used for classification problems.

In [ ]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

In [ ]:
def train(model, loader, optimizer, epoch, device=try_gpu()):
    model.train()
    model = model.to(device)
    total_loss, total_correct, num_labels = 0, 0, 0
    for batch_id, data in enumerate(loader):
        data, target = data[0], data[-1]

        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_labels += data.size(0)

        pred = output.argmax(dim=1, keepdim=True)  # get the index of the max probability
        num_corrects = pred.eq(target.view_as(pred)).sum().item()

        total_correct += num_corrects

        if (batch_id + 1) % 100 == 0:
           print('Train Epoch: {} Batch [{}/{} ({:.0f}%)]\tLoss: {:.6f} Accuracy: {}/{} ({:.0f}%)'.format(
                epoch, (batch_id + 1) * len(data), len(loader.dataset),
                100. * (batch_id + 1) / len(loader), total_loss / num_labels, total_correct, num_labels, 100. * total_correct / num_labels))

    print('Train Epoch: {} Average Loss: {:.6f} Average Accuracy: {}/{} ({:.0f}%)'.format(
         epoch, total_loss / num_labels, total_correct, num_labels, 100. * total_correct / num_labels))

The `test` function will be used to evaluate the performance of the model on both the validation and test splits, depending on the `loader` and `text` parameters.

In [ ]:
def test(model, loader, text='Validation', device=try_gpu()):
    model.eval()
    model = model.to(device)
    total_loss, total_correct, num_labels = 0, 0, 0
    with torch.no_grad():
         for batch_id, data in enumerate(loader):
             data, target = data[0], data[-1]

             data = data.to(device)
             target = target.to(device)

             output = model(data)
             loss = F.cross_entropy(output, target)

             total_loss += loss.item()
             num_labels += data.size(0)

             pred = output.argmax(dim=1, keepdim=True)  # get the index of the max probability
             num_corrects = pred.eq(target.view_as(pred)).sum().item()

             total_correct += num_corrects

    print(text + ' Average Loss: {:.6f} Average Accuracy: {}/{} ({:.0f}%)'.format(
         total_loss / num_labels, total_correct, num_labels, 100. * total_correct / num_labels))

We are now ready to train the model. The `StepLR` learning rate scheduler is used to multiply the learning rate by `gamma` after each epoch.

In [ ]:
lr = 1e-2 #9 min
gamma = 0.7
total_epochs = 1

model = VideoRecognitionModel()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, total_epochs + 1):
    train(model, train_loader, optimizer, epoch)
    test(model, val_loader, text="Validation")
    scheduler.step()

Train Epoch: 1 Batch [400/7366 (5%)]	Loss: 0.991454 Accuracy: 14/400 (4%)
Train Epoch: 1 Batch [800/7366 (11%)]	Loss: 0.987205 Accuracy: 32/800 (4%)
Train Epoch: 1 Batch [1200/7366 (16%)]	Loss: 0.985789 Accuracy: 52/1200 (4%)
Train Epoch: 1 Batch [1600/7366 (22%)]	Loss: 0.985081 Accuracy: 69/1600 (4%)
Train Epoch: 1 Batch [2000/7366 (27%)]	Loss: 0.984656 Accuracy: 94/2000 (5%)
Train Epoch: 1 Batch [2400/7366 (33%)]	Loss: 0.984373 Accuracy: 115/2400 (5%)
Train Epoch: 1 Batch [2800/7366 (38%)]	Loss: 0.984170 Accuracy: 137/2800 (5%)
Train Epoch: 1 Batch [3200/7366 (43%)]	Loss: 0.984019 Accuracy: 158/3200 (5%)
Train Epoch: 1 Batch [3600/7366 (49%)]	Loss: 0.983901 Accuracy: 175/3600 (5%)
Train Epoch: 1 Batch [4000/7366 (54%)]	Loss: 0.983806 Accuracy: 188/4000 (5%)
Train Epoch: 1 Batch [4400/7366 (60%)]	Loss: 0.983729 Accuracy: 207/4400 (5%)
Train Epoch: 1 Batch [4800/7366 (65%)]	Loss: 0.983665 Accuracy: 228/4800 (5%)
Train Epoch: 1 Batch [5200/7366 (71%)]	Loss: 0.983610 Accuracy: 245/5200 (

Finally, we can test the model on the test split.

In [ ]:
test(model, test_loader, text="Test") #2 min

Test Average Loss: 0.983567 Average Accuracy: 179/3234 (6%)


#Exercises

1. Implement ResNet3D-18 from scratch starting from the implementation of ResNet in *Laboratory 4*.

2. Replace the ResNet3D-18 model with the C3D model given in *Figure 8, Lecture 9*, and compare their performance.